# pandas 之七：分组聚合（groupby）

---

## groupby 是什么

数据分析里最经典的动作是 **"分而治之"**：

```
拆分（split） -> 应用（apply） -> 合并（combine）
```

举个例子："**每个岗位的平均工资是多少？**"

1. **拆分**：按 `job_name` 把 500 条员工数据分成若干组
2. **应用**：每组各算一次 `salarys.mean()`
3. **合并**：把各组结果拼成一张新表

这个套路在 SQL 里叫 `GROUP BY`，在 Excel 里叫**数据透视表**。

## 万能公式

```python
df.groupby('分组依据列')['要聚合的列'].聚合函数()
```

**只要记住这一句，80% 的分组需求都能解决。**

## 学习目标

1. 掌握 groupby 的基本语法
2. 会看分组结果（`groups` / `get_group` / `size`）
3. 会多列分组
4. 会用 `agg` 做**多函数、多列、命名**聚合
5. 会遍历分组、用 `transform` 做组内计算


In [ ]:
import pandas as pd

print('pandas 版本：', pd.__version__)


---
## 准备数据

用项目里的 `employee_data.csv`（500 条员工记录）。

> **注意 `encoding='utf-8-sig'`**：这份 CSV 是 Windows 下生成的，带 BOM。
> 不加这个参数，第一个列名会多出一个看不见的 `\ufeff` 字符。


In [ ]:
df = pd.read_csv('employee_data.csv', encoding='utf-8-sig')
print('形状：', df.shape)
print('列名：', list(df.columns))
print()
print(df.head(3))


---
## 一、基本分组：先"看"分组

`groupby()` 本身**不会立刻计算**，它只是"准备好分组方案"。
这叫**惰性求值**，只有在你调用聚合函数时才算。


In [ ]:
grouped = df.groupby('job_name')
print('grouped 对象：', grouped)
print('  <- 它不是一个结果表，而是"待计算的分组对象"')


In [ ]:
# groups：看每个分组的"成员索引"
groups = grouped.groups
print('分组个数：', len(groups))
print('\n各分组的成员索引（只显示前几个）：')
for name, idx in list(groups.items())[:3]:
    print(f'  {name}: {list(idx[:5])} ...共 {len(idx)} 人')


### `size()`：每组有多少条


In [ ]:
size = grouped.size()
print('每个岗位的人数（自动按人数降序排不了，这里是按岗位名）：')
print(size)

print('\n按人数从多到少：')
print(size.sort_values(ascending=False))


### `get_group()`：把某一组"拎出来"看


In [ ]:
sw = grouped.get_group('软件工程师')
print('软件工程师共', len(sw), '人')
print('\n前 3 行：')
print(sw.head(3))

print('\n他们的平均工资：', round(sw['salarys'].mean(), 2))


---
## 二、基本聚合：一行拿到结果


In [ ]:
# 只取一列 -> 得到 Series
avg_by_job = df.groupby('job_name')['salarys'].mean()
print('每位岗位的平均工资（Series）：')
print(avg_by_job.round(2))


### ⚠️ `['列']` 和 `[['列']]` 的区别（重要）

| 写法 | 返回类型 |
|---|---|
| `df.groupby('x')['y'].mean()` | **Series** |
| `df.groupby('x')[['y']].mean()` | **DataFrame** |

**多层方括号 = 保留下"列"这个维度**，所以是 DataFrame。


In [ ]:
s_result = df.groupby('job_name')['salarys'].mean()
df_result = df.groupby('job_name')[['salarys']].mean()

print('单括号 -> ', type(s_result).__name__)
print('双括号 -> ', type(df_result).__name__)
print('\n双括号的可以继续用列名取值：')
print(df_result['salarys'].round(2).head(3))


### 完整流程：算完 + 整理


In [ ]:
job_salary = df.groupby('job_name')[['salarys']].mean()
job_salary['salarys'] = job_salary['salarys'].round(2)   # 保留 2 位
job_salary = job_salary.reset_index()                    # 索引变回普通列
job_salary = job_salary.sort_values(by='salarys', ascending=False)

print('各岗位平均工资排行：')
print(job_salary)


> **注意 `reset_index()` 这一步**：
> groupby 之后，`job_name` 会变成**索引**，
> 想让它变回普通列（后续排序、导出都需要），就要 `reset_index()`。
>
> **这是 groupby 最常忘的一步。**


---
## 三、多列分组

按"岗位 + 年龄"一起分组，会得到**多级索引**的结果。


In [ ]:
multi = df.groupby(['job_name', 'age']).size()
print('前 8 个分组的人数：')
print(multi.head(8))


**多级索引怎么用？**

- `reset_index()` 变成普通列（最省事）
- `unstack()` 把最内层变成列


In [ ]:
multi_df = df.groupby(['job_name', 'age']).size().reset_index(name='人数')
print('reset_index 之后：')
print(multi_df.head(6))


> **`reset_index(name='人数')`** 是个小技巧：
> 因为 `size()` 出来的 Series 没有列名，直接 `reset_index()` 那一列会叫 `0`。
> 用 `name=` 顺便起个名字。


---
## 四、`agg`：一次做多种聚合

前面每次只能算一个指标。真实报表往往要"总和 + 平均 + 最大"一起出。


In [ ]:
# 一个列，多个函数
one_col = df.groupby('job_name')['salarys'].agg(['mean', 'max', 'min', 'count'])
print('工资的多种统计（每列一个函数）：')
print(one_col.round(1).head())


### 不同列用不同函数：传字典


In [ ]:
two_col = df.groupby('job_name').agg({
    'salarys': 'mean',      # 工资求平均
    'age': 'max',           # 年龄取最大
})
print('工资平均 + 年龄最大：')
print(two_col.round({'salarys': 2}).head())


### 一个列要多个函数 + 另一个列一个函数


In [ ]:
multi_agg = df.groupby('job_name').agg({
    'salarys': ['mean', 'max'],
    'age': 'mean',
}).reset_index()
print('列名变成了多级（MultiIndex）：')
print(multi_agg.head())


多级列名有点丑，可以用**命名聚合**解决。


### ⭐ 命名聚合（推荐写法）

这是 pandas 里**最优雅**的聚合写法：**自己指定结果列名**。


In [ ]:
named = df.groupby('job_name').agg(
    平均工资=('salarys', 'mean'),
    最高工资=('salarys', 'max'),
    员工数=('employ_id', 'count'),
    平均年龄=('age', 'mean'),
).reset_index()

named['平均工资'] = named['平均工资'].round(2)
named['平均年龄'] = named['平均年龄'].round(1)
print('命名聚合的结果（列名清爽）：')
print(named.sort_values('平均工资', ascending=False))


**命名聚合的语法**：

```python
df.groupby('分组列').agg(
    新列名=('原列名', '聚合函数'),
    另一个新列名=('原列名', '聚合函数'),
)
```

**好处**：
- 列名自己定，不用再改 MultiIndex
- 顺序和你写的一致
- 可读性最强，**推荐在正式代码里用这种**


### 自定义聚合函数

内置函数不够用时，可以传自己的函数（比如"极差"、"中位数"）。


In [ ]:
def range_func(x):
    """极差 = 最大值 - 最小值"""
    return x.max() - x.min()


def median_func(x):
    """中位数"""
    return x.median()


custom = df.groupby('job_name').agg(
    极差=('salarys', range_func),
    中位数=('salarys', median_func),
).reset_index()
print('自定义聚合（极差 / 中位数）：')
print(custom.round(1).head(6))


> **传函数时注意**：写 `('salarys', range_func)` 就好，**不要加括号写 `range_func()`**。
> 加括号会"当场调用"，而我们需要的是把**函数本身**交给 pandas 去逐组调用。
> 这是很典型的一个错。


In [ ]:
# 也可以直接用 lambda（适合只有一行的简单逻辑）
lam = df.groupby('job_name')['salarys'].agg(lambda x: x.max() - x.min())
print('用 lambda 算极差：')
print(lam.round(1).head())


---
## 五、遍历分组

需要在每组里做**复杂自定义操作**时，就直接遍历。


In [ ]:
print('遍历所有分组，打印每组的基本信息：')
print('=' * 50)

for job_name, group_data in df.groupby('job_name'):
    print(f'\n【{job_name}】')
    print(f'  人数：{len(group_data)}')
    print(f'  平均工资：{round(group_data["salarys"].mean(), 1)}')
    print(f'  年龄范围：{group_data["age"].min()} ~ {group_data["age"].max()}')


**遍历分组的语法**：

```python
for 分组名, 该组的数据 in df.groupby('列名'):
    ...
```

- 第一个变量是**分组名**（该组的标签值）
- 第二个变量是**子 DataFrame**（结构和原表一样）


In [ ]:
print('第一组数据的类型：', type(list(df.groupby('job_name'))[0][1]).__name__)
print('  <- 是 DataFrame，可以照常用 .mean()、.sort_values() 等')


---
## 六、`transform`：把组内统计结果"贴回"每一行

`agg` 会把每组压成一行，`transform` 则**保持原行数**，
把组内的统计值广播回每一行。

**用途**：算"每个人的工资 / 该岗位平均工资"这种**组内相对值**。


In [ ]:
d = df[['job_name', 'salarys']].head(6).copy()

d['组内平均'] = d.groupby('job_name')['salarys'].transform('mean')
d['与组均值的差'] = d['salarys'] - d['组内平均']
d['组内排名'] = d.groupby('job_name')['salarys'].rank(ascending=False)

print(d.round(2))


**看第 1、4 行**：它们都是"项目经理"，所以"组内平均"一样，
但各自的"与组均值的差"不同。**这就是 `transform` 的典型用途。**


### `agg` vs `transform` 对比

| | 行数 | 用途 |
|---|---|---|
| `agg` | **压缩**（每组一行） | 出汇总报表 |
| `transform` | **不变**（和原表一样多行） | 算组内相对指标 |
| `filter` | 变少（筛掉整组） | 只保留满足条件的组 |


In [ ]:
# filter：只保留"平均工资 > 20000"的岗位的所有人
high = df.groupby('job_name').filter(lambda g: g['salarys'].mean() > 20000)
print('高薪岗位的记录数：', len(high))
print('涉及岗位：', sorted(high['job_name'].unique()))


---
## 七、实用：一份"部门/岗位薪酬分析报告"

把上面学的串起来，做一个完整的分析。


In [ ]:
report = df.groupby('job_name').agg(
    人数=('employ_id', 'count'),
    平均工资=('salarys', 'mean'),
    工资中位数=('salarys', 'median'),
    最高工资=('salarys', 'max'),
    最低工资=('salarys', 'min'),
    平均年龄=('age', 'mean'),
).reset_index()

# 派生指标：极差、平均工资排名
report['工资极差'] = report['最高工资'] - report['最低工资']
report = report.round({'平均工资': 0, '工资中位数': 0, '平均年龄': 1})
report = report.sort_values('平均工资', ascending=False).reset_index(drop=True)

print('=' * 70)
print('各岗位薪酬分析报告')
print('=' * 70)
print(report.to_string(index=False))


In [ ]:
print('\n=== 关键结论 ===')
top = report.iloc[0]
bottom = report.iloc[-1]
print(f'· 平均工资最高：{top["job_name"]}（{top["平均工资"]:.0f} 元）')
print(f'· 平均工资最低：{bottom["job_name"]}（{bottom["平均工资"]:.0f} 元）')
print(f'· 差距：{top["平均工资"] / bottom["平均工资"]:.2f} 倍')
print(f'· 全员平均工资：{df["salarys"].mean():.0f} 元')
print(f'· 全员平均年龄：{df["age"].mean():.1f} 岁')


---
## 八、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 忘记 `reset_index()` | 分组列还在索引里，导出后少一列 | 聚合后接 `.reset_index()` |
| 以为 `groupby` 会立刻算 | 打印出来是个对象 | 它惰性求值，要接聚合函数 |
| `('salarys', func())` 加了括号 | 报错或结果不对 | 传**函数本身**，不加括号 |
| `['列']` / `[['列']]` 混淆 | 返回 Series / DataFrame 不同 | 需要 DataFrame 就加双层括号 |
| `size()` 结果列名是 0 | 不好看 | `.reset_index(name='人数')` |
| 读带 BOM 的 CSV 列名怪 | 分组报 `KeyError` | `encoding='utf-8-sig'` |
| 多列分组后不知道怎么用 | 多级索引太绕 | `.reset_index()` 摊平 |
| 用 `agg` 算"组内相对值" | 行数对不上原表 | 该用 `transform` |
| `filter` 和布尔筛选搞混 | 结果不符合预期 | `filter` 是**按组**筛，保留整组 |

---

## 九、动手练习

**练习 1**：按 `job_name` 分组，算出每个岗位的**人数**和**最高工资**。

**练习 2**：算出每个岗位的**工资中位数**（用命名聚合，结果列叫 `工资中位数`）。

**练习 3**：按 `job_name` 和 `age` 双列分组，统计每组的平均工资。

**练习 4**：找出"平均年龄最大"的岗位（提示：聚合后 `sort_values` 或 `idxmax`）。

**练习 5**：给每个员工标出"他所在岗位的平均工资"，以及"他的工资比岗位平均高多少"（用 `transform`）。

**练习 6**：只保留"人数 >= 40"的岗位的数据（用 `filter`）。

---

**答案思路（先自己做）**

```python
# 1
print(df.groupby('job_name').agg(
    人数=('employ_id', 'count'),
    最高工资=('salarys', 'max'),
).reset_index())

# 2
print(df.groupby('job_name').agg(
    工资中位数=('salarys', 'median')
).reset_index())

# 3
print(df.groupby(['job_name', 'age'])['salarys'].mean().reset_index().head())

# 4
r = df.groupby('job_name')['age'].mean().reset_index()
print(r.sort_values('age', ascending=False).head(1))

# 5
d = df[['job_name', 'salarys']].copy()
d['岗位均值'] = d.groupby('job_name')['salarys'].transform('mean')
d['高出多少'] = d['salarys'] - d['岗位均值']

# 6
print(len(df.groupby('job_name').filter(lambda g: len(g) >= 40)))
```
